# Issue #2: Collect EV Charging Sessions

**Output:** `acn_raw_sessions.csv`, `collection_log.csv`

This notebook repeats the Issue #1 connection and then collects all sessions.

**How collection works (and why).** Requesting deep page numbers (page 140 and beyond) makes the ACN server return HTTP 502 errors, and resuming a page loop by hand appended the same pages twice. This notebook instead requests **one week of sessions at a time** using the API's `where` filter, so every request stays at a shallow page. Each finished week is saved to `acn_windows/`, so the notebook can be stopped and re-run at any time: finished weeks are skipped and no session can be collected twice. The result is de-duplicated and checked against the total the API reports.

**Run order:** run all cells. If the collection cell stops on an error, run it again — it continues where it stopped.

## Issue #1: API connection

In [3]:
import json
import time
from datetime import datetime, timedelta, timezone
from getpass import getpass
from pathlib import Path

import pandas as pd
import requests

URL = "https://ev.caltech.edu/api/v1/sessions/caltech"
API_TOKEN = getpass("Enter your ACN API token: ")      # never printed or saved


def api_get(params, attempts=6, timeout=90):
    """GET with retries and back-off (5, 10, 20, 40, 60, 60 s). Raises after the last attempt."""
    last = None
    for attempt in range(1, attempts + 1):
        try:
            r = requests.get(URL, params=params, auth=(API_TOKEN, ""), timeout=timeout)
            if r.status_code == 200:
                return r.json()
            last = f"HTTP {r.status_code}"
        except requests.exceptions.RequestException as e:
            last = type(e).__name__
        print(f"    {last} (attempt {attempt}/{attempts})")
        if attempt < attempts:
            time.sleep(min(60, 5 * 2 ** (attempt - 1)))
    raise RuntimeError(f"Request failed after {attempts} attempts ({last}). Re-run this cell to resume.")


first = api_get({"max_results": 25})
API_TOTAL = first["_meta"]["total"]
print("ACN API connection successful.")
print("Response keys:", list(first.keys()))
print("Fields per session:", list(first["_items"][0].keys()))
print("Sessions reported by the API:", f"{API_TOTAL:,}")

Enter your ACN API token:  ········


ACN API connection successful.
Response keys: ['_items', '_links', '_meta']
Fields per session: ['_id', 'userInputs', 'sessionID', 'stationID', 'spaceID', 'siteID', 'clusterID', 'connectionTime', 'disconnectTime', 'kWhDelivered', 'doneChargingTime', 'timezone', 'userID']
Sessions reported by the API: 31,424


**API fields:** `_id` (record id), `sessionID`, `stationID`, `spaceID`, `siteID`, `clusterID`, `connectionTime`, `disconnectTime`, `kWhDelivered`, `doneChargingTime`, `timezone`, `userID` (when available), `userInputs` (when available).

## Issue #2: Collect all sessions by weekly time window

In [4]:
WINDOW_DIR = Path("acn_windows")
WINDOW_DIR.mkdir(exist_ok=True)
WINDOW = timedelta(days=7)
START = datetime(2018, 4, 23, tzinfo=timezone.utc)         # Monday before the first session (25 Apr 2018)
MAX_PAGE_SIZE = 100


def http_date(dt):
    """Date format used by the API, e.g. 'Mon, 23 Apr 2018 00:00:00 GMT'."""
    return dt.astimezone(timezone.utc).strftime("%a, %d %b %Y %H:%M:%S GMT")


def parse_api_time(s):
    return datetime.strptime(s, "%a, %d %b %Y %H:%M:%S GMT").replace(tzinfo=timezone.utc)


def fetch_window(start, end):
    """All sessions with start <= connectionTime < end (follows pages inside the window)."""
    where = f'connectionTime>="{http_date(start)}" and connectionTime<"{http_date(end)}"'
    rows, page, expected = [], 1, None
    while True:
        data = api_get({"where": where, "max_results": MAX_PAGE_SIZE, "page": page})
        items = data.get("_items", [])
        expected = data.get("_meta", {}).get("total", expected)
        rows.extend(items)
        if not items or "next" not in data.get("_links", {}):
            break
        page += 1
        time.sleep(0.5)
    # Safety check: the filter must really have been applied
    outside = [r for r in rows if not (start <= parse_api_time(r["connectionTime"]) < end)]
    if outside:
        raise RuntimeError(f"The API returned {len(outside)} sessions outside {start:%Y-%m-%d}..{end:%Y-%m-%d}; "
                           "the 'where' filter was not applied. Stop and report this.")
    if expected is not None and len(rows) != expected:
        raise RuntimeError(f"Week {start:%Y-%m-%d}: got {len(rows)} of {expected} sessions. Re-run to retry.")
    return rows


def collected_so_far():
    return sum(json.loads(p.read_text())["n"] for p in WINDOW_DIR.glob("*.json"))


now = datetime.now(timezone.utc)
done = collected_so_far()
print(f"Already collected in earlier runs: {done:,} / {API_TOTAL:,}\n")

w = START
empty_streak = 0
while w < now and done < API_TOTAL:
    w_end = w + WINDOW
    f = WINDOW_DIR / f"{w:%Y-%m-%d}.json"
    if f.exists():
        n = json.loads(f.read_text())["n"]
    else:
        rows = fetch_window(w, w_end)
        f.write_text(json.dumps({"start": http_date(w), "end": http_date(w_end), "n": len(rows), "items": rows}))
        n = len(rows)
        done += n
        print(f"Week {w:%Y-%m-%d}: {n:4d} sessions | total {done:,} / {API_TOTAL:,}")
        time.sleep(1)
    empty_streak = empty_streak + 1 if n == 0 else 0
    w = w_end

print(f"\nCollection loop finished: {done:,} sessions in {len(list(WINDOW_DIR.glob('*.json')))} weekly files.")
if done < API_TOTAL:
    print("Fewer sessions than the API total - re-run this cell; if the count does not change, see the check below.")

Already collected in earlier runs: 0 / 31,424

Week 2018-04-23:  210 sessions | total 210 / 31,424
Week 2018-04-30:  356 sessions | total 566 / 31,424
Week 2018-05-07:  380 sessions | total 946 / 31,424
Week 2018-05-14:  426 sessions | total 1,372 / 31,424
Week 2018-05-21:  427 sessions | total 1,799 / 31,424
Week 2018-05-28:  378 sessions | total 2,177 / 31,424
Week 2018-06-04:  432 sessions | total 2,609 / 31,424
Week 2018-06-11:  449 sessions | total 3,058 / 31,424
Week 2018-06-18:  430 sessions | total 3,488 / 31,424
Week 2018-06-25:  438 sessions | total 3,926 / 31,424
Week 2018-07-02:  376 sessions | total 4,302 / 31,424
Week 2018-07-09:  483 sessions | total 4,785 / 31,424
Week 2018-07-16:  479 sessions | total 5,264 / 31,424
Week 2018-07-23:  502 sessions | total 5,766 / 31,424
Week 2018-07-30:  526 sessions | total 6,292 / 31,424
Week 2018-08-06:  494 sessions | total 6,786 / 31,424
Week 2018-08-13:  566 sessions | total 7,352 / 31,424
Week 2018-08-20:  537 sessions | total 7,

In [5]:
# Combine the weekly files, remove any duplicates, validate, and save the raw dataset
rows = [r for p in sorted(WINDOW_DIR.glob("*.json")) for r in json.loads(p.read_text())["items"]]
raw_df = pd.DataFrame(rows)
n_rows = len(raw_df)
n_dup = int(raw_df["_id"].duplicated().sum())
raw_df = raw_df.drop_duplicates("_id").reset_index(drop=True)

conn = pd.to_datetime(raw_df["connectionTime"], format="%a, %d %b %Y %H:%M:%S GMT", utc=True)
raw_df = raw_df.iloc[conn.argsort(kind="stable")].reset_index(drop=True)     # chronological order
conn = conn.sort_values().reset_index(drop=True)

# Weekly counts and any long runs of empty days (would indicate missing data)
days = conn.dt.tz_convert("America/Los_Angeles").dt.normalize().dt.tz_localize(None)
all_days = pd.date_range(days.min(), days.max(), freq="D")
per_day = days.value_counts().reindex(all_days, fill_value=0)
zero = per_day.eq(0)
runs = zero.ne(zero.shift()).cumsum()
gap_runs = per_day[zero].groupby(runs[zero]).apply(lambda s: (s.index.min().date(), s.index.max().date(), len(s)))
long_gaps = [g for g in gap_runs if g[2] >= 7]

log = pd.DataFrame([{"week_start": json.loads(p.read_text())["start"], "sessions": json.loads(p.read_text())["n"]}
                    for p in sorted(WINDOW_DIR.glob("*.json"))])
log.to_csv("collection_log.csv", index=False)

print(f"Rows in weekly files: {n_rows:,} | duplicates removed: {n_dup} | unique sessions: {len(raw_df):,}")
print(f"API total: {API_TOTAL:,} | coverage: {len(raw_df) / API_TOTAL:.2%}")
print(f"Date range (local): {days.min().date()} to {days.max().date()}")
print("Runs of 7+ days with no sessions:", long_gaps if long_gaps else "none")

assert raw_df["_id"].is_unique, "duplicate _id values remain"
assert raw_df["sessionID"].is_unique, "duplicate sessionID values remain"
raw_df.to_csv("acn_raw_sessions.csv", index=False)
print("\nSaved acn_raw_sessions.csv:", raw_df.shape)

Rows in weekly files: 31,424 | duplicates removed: 0 | unique sessions: 31,424
API total: 31,424 | coverage: 100.00%
Date range (local): 2018-04-25 to 2021-09-13
Runs of 7+ days with no sessions: [(datetime.date(2020, 8, 4), datetime.date(2020, 11, 17), 106)]

Saved acn_raw_sessions.csv: (31424, 13)


In [6]:
# Issue #2 summary, generated from the data (so it can never disagree with the outputs above)
from IPython.display import Markdown, display
complete = len(raw_df) == API_TOTAL
gap_txt = ("; ".join(f"{a} to {b} ({n} days)" for a, b, n in long_gaps)) if long_gaps else "none"
summary2 = f"""## Issue #2: Collected EV Charging Sessions

- **Method:** authenticated REST requests, one week of `connectionTime` per request (`where` filter), up to {MAX_PAGE_SIZE} records per page, automatic retries with back-off, weekly checkpoint files.
- **Collected:** {len(raw_df):,} unique sessions of the {API_TOTAL:,} reported by the API ({len(raw_df)/API_TOTAL:.1%}){' — the complete dataset' if complete else ''}.
- **Coverage:** {days.min().date()} to {days.max().date()} (local time); runs of 7+ days without sessions: {gap_txt}.
- **Duplicates:** {n_dup} duplicate records found across weekly files and removed; `_id` and `sessionID` are unique.
- **Fields:** {raw_df.shape[1]} ({', '.join(raw_df.columns)}).
- **Output:** `acn_raw_sessions.csv`; per-week counts in `collection_log.csv`.
"""
display(Markdown(summary2))

## Issue #2: Collected EV Charging Sessions

- **Method:** authenticated REST requests, one week of `connectionTime` per request (`where` filter), up to 100 records per page, automatic retries with back-off, weekly checkpoint files.
- **Collected:** 31,424 unique sessions of the 31,424 reported by the API (100.0%) — the complete dataset.
- **Coverage:** 2018-04-25 to 2021-09-13 (local time); runs of 7+ days without sessions: 2020-08-04 to 2020-11-17 (106 days).
- **Duplicates:** 0 duplicate records found across weekly files and removed; `_id` and `sessionID` are unique.
- **Fields:** 13 (_id, userInputs, sessionID, stationID, spaceID, siteID, clusterID, connectionTime, disconnectTime, kWhDelivered, doneChargingTime, timezone, userID).
- **Output:** `acn_raw_sessions.csv`; per-week counts in `collection_log.csv`.
